In [1]:
import os
from typing import Optional

import numpy as np
import onnx
import torch
import transformers

/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from generateModels import (
    fix_onnx_fp16,
    generate_input_files,
    generate_npi,
    parse_args,
    run_model_on_aic,
    run_model_on_ort,
    save_onnx,
)

In [3]:
seq_len = 128
model_name = 'gpt2'
input_str = 'Hello, How are you?'
model_path = 'Gpt2Small'
model_base_name = model_name

In [4]:
# Load tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model (always Causal model)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, use_cache=True, return_dict=False
)
model.eval()
assert not model.config.is_decoder, "Model is an encoder-decoder model"

# Tracing requires disabling grad for all parameters
for param in model.parameters():
    param.requires_grad = False

# Preprocess inputs
inputs = tokenizer(input_str, return_tensors="pt", padding="max_length", max_length=seq_len)
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

# Model wrapper for tracing
def model_fn(input_ids, attention_mask, position_ids=None, *past_key_values):
    if past_key_values:
        pkv = []
        for i in range(0, len(past_key_values), 2):
            pkv.append(tuple(past_key_values[i : i + 2]))
        past_key_values = tuple(pkv)
    else:
        past_key_values = None
    logits, past_key_values = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
        past_key_values=past_key_values,
    )
    outputs = [logits]
    for key_values in past_key_values:
        outputs.extend(key_values)
    return tuple(outputs)

# Trace model for initial iteration
ts_init = torch.jit.trace(model_fn, (input_ids, attention_mask))

# Prepare inputs for next iterations
batch_size = input_ids.shape[0]
logits, *pkv = ts_init(input_ids, attention_mask)
input_len = attention_mask.sum(1, keepdim=True)
nxt_token = torch.argmax(
    logits[torch.arange(batch_size), input_len.squeeze(1) - 1], 1, keepdim=True
)
attention_mask_2 = torch.cat(
    [attention_mask, torch.ones((batch_size, 1), dtype=torch.int64)], 1
)

# Trace model for next iterations
ts_iter = torch.jit.trace(model_fn, (nxt_token, attention_mask_2, input_len, *pkv))
logits_2, *pkv_2 = ts_iter(nxt_token, attention_mask_2, input_len, *pkv)

# Scripting for loop model
@torch.jit.script
def ts_loop(
    input_ids: torch.LongTensor,
    attention_mask: torch.LongTensor,
    unroll_tensor: torch.LongTensor,
):
    batch_size = input_ids.shape[0]
    logits, *pkv = ts_init(input_ids, attention_mask)
    input_len = attention_mask.sum(1, keepdim=True)
    nxt_token = torch.argmax(logits[torch.arange(batch_size), input_len.squeeze(1) - 1], -1, keepdim=True)
    generated_ids = nxt_token

    for i in range(unroll_tensor.size(0) - 1):
        attention_mask = torch.cat(
            [attention_mask, torch.ones((batch_size, 1), dtype=torch.int64)], 1
        )
        logits, *pkv = ts_iter(nxt_token, attention_mask, input_len + i, *pkv)
        nxt_token = torch.argmax(logits, 2)
        generated_ids = torch.cat([generated_ids, nxt_token], 1)

    return generated_ids



/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/transformers/models/gpt2/modeling_gpt2.py:806: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if batch_size <= 0:
/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/torch/jit/annotations.py:310: UserWarning: TorchScript will treat type annotations of Tensor dtype-specific subtypes as if they are normal Tensors. dtype constraints are not enforced in compilation either.
  warnings.warn("TorchScript will treat type annotations of Tensor "


In [6]:
# Run PyTorch inference
inputs["unroll_tensor"] = torch.arange(5)
pt_outputs = ts_loop(**inputs)
output_names = ["generated_ids"]

# Export and simplify ONNX model
gen_models_path = f"{model_path}/generatedModels"
os.makedirs(gen_models_path, exist_ok=True)
torch.onnx.export(
    ts_loop,
    (
        inputs["input_ids"],
        inputs["attention_mask"],
        inputs["unroll_tensor"],
    ),
    f"{gen_models_path}/{model_base_name}_raw.onnx",
    input_names=["input_ids", "attention_mask", "unroll_tensor"],
    output_names=output_names,
    opset_version=14,
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence"},
        "attention_mask": {0: "batch_size", 1: "sequence"},
        "unroll_tensor": {0: "unroll_len"},
    },
)

# Simplifier might fail if the model is bigger than 2GB, so we're not running it

# Infer shapes
onnx.shape_inference.infer_shapes_path(
    f"{gen_models_path}/{model_base_name}_raw.onnx",
    f"{gen_models_path}/{model_base_name}_inf.onnx",
    True,
    True,
    True,
)

# Save as single tensors file model
save_onnx(
    onnx.load(f"{gen_models_path}/{model_base_name}_inf.onnx"), gen_models_path, model_base_name
)
fp32_model_name = model_base_name

# Run ONNXRT inference
input_names, ort_outputs = run_model_on_ort(
    f"{gen_models_path}/{fp32_model_name}.onnx", inputs, output_names
)
print("PyTorch vs. ONNXRT (MAD):")
pto = pt_outputs

print("Pytorch outputs ----------------------------------------")
for i, output in enumerate(pto):
    print(tokenizer.decode(output, skip_special_tokens=True))

print("--------------------------------------------------------")

for oname, orto in zip(output_names, ort_outputs):
    print(oname, np.abs(pto.detach().numpy() - orto).max())


print("Onnx runtime outputs ----------------------------------------")

outputs = [None for _ in range(len(ort_outputs[0]))]
for i in range(len(ort_outputs[0])):
    outputs[i] = torch.from_numpy(ort_outputs[0][i])
print(outputs)
for i, output in enumerate(outputs):
    print(tokenizer.decode(output, skip_special_tokens=True))
print("--------------------------------------------------------")



============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

PyTorch vs. ONNXRT (MAD):
Pytorch outputs ----------------------------------------


I'm a
--------------------------------------------------------
generated_ids 0
Onnx runtime outputs ----------------------------------------
[tensor([ 198,  198,   40, 1101,  257])]


I'm a
--------------------------------------------------------


In [7]:
# Fix onnx for FP16
fp16_model_name = fix_onnx_fp16(
    inputs, output_names, ort_outputs, gen_models_path, fp32_model_name
)



Found constants out of FP16 range, clipped to FP16 range


In [8]:
# Generate NPI yaml if needed
mixed_precision = False
# Generate inputFiles
input_list_file = f"{model_path}/input_list.txt"
generate_input_files(f"{model_path}/inputFiles", input_names, inputs, input_list_file)


In [9]:
# onnx dynamic shape
onnx_symbol_defs = {
    "batch_size": inputs["input_ids"].shape[0],
    "sequence": inputs["input_ids"].shape[1],
    "unroll_len": inputs["unroll_tensor"].shape[0],
}

# AICOutputs
write_output_dir = f"{model_path}/AICOutputs"
os.makedirs(f"{write_output_dir}/FP32", exist_ok=True)
os.makedirs(f"{write_output_dir}/FP16", exist_ok=True)

# Run on AIC in FP32
assert run_model_on_aic(
    f"{gen_models_path}/{fp32_model_name}.onnx",
    onnx_symbol_defs=onnx_symbol_defs,
    input_list_file=input_list_file,
    convert_to_fp16=False,
    write_output_dir=f"{write_output_dir}/FP32",
), "Compilation failed"

# Run on AIC in FP16
assert run_model_on_aic(
    f"{gen_models_path}/{fp16_model_name}.onnx",
    onnx_symbol_defs=onnx_symbol_defs,
    input_list_file=input_list_file,
    convert_to_fp16=not mixed_precision,
    node_precision_info=npi_path if mixed_precision else False,
    write_output_dir=f"{write_output_dir}/FP16",
), "Compilation failed"

# Verify outputs
print("ONNXRT vs. AIC (MAD)")
for oname, orto in zip(output_names, ort_outputs):
    aico32 = np.fromfile(
        f"{write_output_dir}/FP32/{oname}-activation-0-inf-0.bin", orto.dtype
    ).reshape(orto.shape)
    diff32 = np.abs(orto - aico32).max()
    print(oname, "FP32:", diff32)

    aico16 = np.fromfile(
        f"{write_output_dir}/FP16/{oname}-activation-0-inf-0.bin", orto.dtype
    ).reshape(orto.shape)
    diff16 = np.abs(orto - aico16).max()
    print(oname, "FP16:", diff16)


print("aico32 runtime outputs ----------------------------------------")
print(aico32[0])
aico32_ndarr = aico32[0].astype(np.int64)
print(aico32_ndarr)
outputs = [None for _ in range(len(aico32_ndarr))]

#for i in range(len(aico32_ndarr)):
outputs = torch.from_numpy(aico32_ndarr)
print(outputs)
for i, output in enumerate(outputs):
    print(tokenizer.decode(output, skip_special_tokens=True))
    
print("--------------------------------------------------------")

print("aico16 runtime outputs ----------------------------------------")
print(aico16[0])
aico32_ndarr = aico16[0].astype(np.int64)
print(aico32_ndarr)
outputs = [None for _ in range(len(aico32_ndarr))]

#for i in range(len(aico32_ndarr)):
outputs = torch.from_numpy(aico32_ndarr)
print(outputs)
for i, output in enumerate(outputs):
    print(tokenizer.decode(output, skip_special_tokens=True))
print("--------------------------------------------------------")

Running compiler: /opt/qti-aic/exec/qaic-exec -m=Gpt2Small/generatedModels/gpt2.onnx -aic-hw -aic-hw-version=2.0 -onnx-define-symbol=batch_size,1 -onnx-define-symbol=sequence,128 -onnx-define-symbol=unroll_len,5 -input-list-file=Gpt2Small/input_list.txt -write-output-dir=Gpt2Small/AICOutputs/FP32
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Reading ONNX Model from Gpt2Small/generatedModels/gpt2.onnx
Compile started ............... 
Compiling model with FP32 precision.


QAIC_ERROR: 
Error message: Loop operator M input must be a constant.
Unable to AddNodesToGraphFromModel


QAICException:

AssertionError: Compilation failed

In [11]:
!/opt/qti-aic/dev/python/qaic-env/bin/pip list | grep "torch"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
torch                        2.0.1
torchaudio                   0.9.1
torchtext                    0.5.0
torchvision                  0.15.1


In [ ]:
!which python

In [ ]:
!pip uninstall torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Found existing installation: torch 2.0.0
Uninstalling torch-2.0.0:
  Would remove:
    /usr/local/bin/convert-caffe2-to-onnx
    /usr/local/bin/convert-onnx-to-caffe2
    /usr/local/bin/torchrun
    /usr/local/lib/python3.8/dist-packages/functorch/*
    /usr/local/lib/python3.8/dist-packages/nvfuser/*
    /usr/local/lib/python3.8/dist-packages/torch-2.0.0.dist-info/*
    /usr/local/lib/python3.8/dist-packages/torch/*
    /usr/local/lib/python3.8/dist-packages/torchgen/*
Proceed (y/n)? 